In [0]:
# %pip install openai>=1.56.0
# %pip install azure-keyvault-secrets azure-identity
# %restart_python

In [0]:
from openai import AsyncAzureOpenAI
import asyncio
import mlflow
import json
import re
import pandas as pd
import ast
import numpy as np
import os
from datetime import datetime
from copy import deepcopy
from scipy.stats import entropy, mode
from sklearn.metrics import f1_score
from tqdm.asyncio import tqdm
from sklearn.metrics import f1_score
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

mlflow.openai.autolog(disable=True)
mlflow.autolog(disable=True)

# allows asyncio inside main
import nest_asyncio
nest_asyncio.apply()

In [0]:
# # Primary endpoint
API_KEY = 
client = AsyncAzureOpenAI(
                api_version="2024-10-21",
                azure_endpoint="https://oai-rtai-dev-aoai-east-001.openai.azure.com/",
                api_key=API_KEY,
            )





In [0]:
def get_system_prompt() -> str:
    """Reads and returns the system prompt from prompt.md."""
    try:
        with open("./prompts/NPS_prompt.md", "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return "Error: prompt.md not found. Please ensure the file exists in the correct directory."

def get_schema() -> dict:
    """Returns the JSON schema wrapped for OpenAI Structured Outputs."""
    schema_definition = {
        "name": "sentiment_analysis",
        "description": "Predicts call sentiment for Spectrum Customer Solutions based on the NPS Rubric.",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "class": {
                    "type": "string",
                    "enum": ["Promoter", "Passive", "Detractor", "None"],
                    "description": "The categorical NPS label."
                },
                "score": {
                    "type": "integer",
                    "description": "The numerical score from -1 to 10."
                },
                "rationale": {
                    "type": "string",
                    "description": "Explanation of reasoning, including how background talk/hold time was handled and justification using the rubric."
                },
                "citations": {
                    "type": "array",
                    "items": {
                        "type": "integer"
                    },
                    "description": "A list of the specific transcript line numbers that served as primary evidence."
                }
            },
            "required": ["class", "score", "rationale", "citations"],
            "additionalProperties": False
        }
    }
   
    return {
        "type": "json_schema",
        "json_schema": schema_definition
    }

In [0]:
# LLM class
async def execute_completion(config: dict, 
                             user_input: str, 
                             client: AsyncAzureOpenAI = client):
    response_format = config["response_format"]
    system_prompt = config["system_prompt"]
    reasoning_effort = config["reasoning_effort"]
    model_name = config["model_name"]

    n_retries = config['n_retries']
    for attempt in range(n_retries):
        try:
            response = await client.chat.completions.with_raw_response.create(
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_input}
                ],
                response_format=response_format,
                reasoning_effort=reasoning_effort,
                model=model_name,
                timeout=60.0
            )
            headers = response.headers
            parsed = response.parse()
            content = parsed.choices[0].message.content

            # capture debug info
            debug_info = str({
                "rate_limits": {
                    "tokens_left": headers.get('x-ratelimit-remaining-tokens'),
                    "requests_left": headers.get('x-ratelimit-remaining-requests'),
                },
                "usage": {
                    "prompt_tokens": parsed.usage.prompt_tokens,
                    "completion_tokens": parsed.usage.completion_tokens,
                    "cached_tokens": getattr(parsed.usage.prompt_tokens_details, 'cached_tokens', 0),
                    "total_tokens": parsed.usage.total_tokens,
                },
                "metadata": {
                    "model": parsed.model,
                    "finish_reason": parsed.choices[0].finish_reason,
                    "request_id": headers.get('x-request-id'),
                    "processing_ms": headers.get('openai-processing-ms')
                }
            })

            return content, debug_info
        
        except Exception as e:
            print(f"Error occurred: {e}")
            fallback_content = json.dumps({
                "class": "error",
                "certainty": "error",
                "rationale": "error",
                "citations": ["error"]
            })
            fallback_debug_info = str({
                "rate_limits": {"tokens_left": 0, "requests_left": 0},
                "usage": {"prompt_tokens": 0, "completion_tokens": 0, "cached_tokens": 0, "total_tokens": 0},
                "metadata": {"model": 0, "finish_reason": 0, "request_id": 0, "processing_ms": 0}
            })
            return fallback_content, fallback_debug_info

In [0]:

async def async_driver_semaphore(config: dict,
                 d: str,
                 sem: asyncio.Semaphore = None,
                 client: AsyncAzureOpenAI = client) -> pd.DataFrame:
    
    async with sem:
        return await execute_completion(config, d, client)

async def async_driver(config: dict,
                 df: pd.DataFrame, 
                 client: AsyncAzureOpenAI = client) -> pd.DataFrame:
    
    # cast our data to a list
    data = df['Text'].tolist()

    # initialize semaphore
    sem = asyncio.Semaphore(config['semaphore_value'])

    # make the task list
    tasks = [
        async_driver_semaphore(config, d, sem, client)
        for d in data
    ]

    # unpack the list when passing to 
    results = await tqdm.gather(*tasks)
    completions, debug = zip(*results) 

    # enrich df with responses
    df['predictions'] = list(completions)
    df['debug_info'] = list(debug)

    return df

def extract_token_data(df):
   
    # 1. Convert the 'debug_info' strings back into real dictionaries
    df['debug_dict'] = df['debug_info'].apply(ast.literal_eval)

    # 2. Extract the specific usage values
    df['prompt_tokens'] = df['debug_dict'].apply(lambda x: x['usage']['prompt_tokens'])
    df['cached_tokens'] = df['debug_dict'].apply(lambda x: x['usage']['cached_tokens'])
    df['completion_tokens'] = df['debug_dict'].apply(lambda x: x['usage']['completion_tokens'])
    df['total_tokens'] = df['debug_dict'].apply(lambda x: x['usage']['total_tokens'])

    # 3. Calculate cache rate per row
    df['cache_rate'] = df['cached_tokens'] / df['prompt_tokens']

    # 4. Get the final averages
    avg_prompt_tokens = df['prompt_tokens'].mean()
    avg_cache_rate = df['cache_rate'].mean()
    avg_completion_tokens = df['completion_tokens'].mean()
    avg_total_tokens = df['total_tokens'].mean()

    # print(f"Avg Tokens per Row: {avg_prompt_tokens:.2f}")
    # print(f"Avg Prompt Cache Rate: {avg_cache_rate:.2%}")
    # print(f"Avg Completion Tokens per Row: {avg_completion_tokens:.2f}")
    # print(f"Avg Total Tokens per Row: {avg_total_tokens:.2f}")

    return df

In [0]:
def extract_data(df):
    parsed = df['predictions'].apply(json.loads)

    df['class'] = parsed.apply(lambda x: x.get('class'))
    df['score'] = parsed.apply(lambda x: x.get('score'))
    df['rationale'] = parsed.apply(lambda x: x.get('rationale'))
    df['citations'] = parsed.apply(lambda x: x.get('citations'))

    # drop classes not included in original analysis
    excluded_labels = ['']
    try:
        df = df[~df['label'].isin(excluded_labels)].copy()
    except Exception as e:
        print(f"Error filtering excluded_labels: {e}")

    # Standardize label values using mapping dictionary
    mapping = {
    }

    try:
        df['label'] = df['label'].apply(lambda val: mapping.get(val, val))
    except Exception as e:
        print(f"Error mapping label values: {e}")

    try:        
        df = extract_token_data(df)
    except Exception as e:
        print(f'Error in extract_token_data: {e}')

    return df

In [0]:
def eval(df_original, config):
    
    df_run = asyncio.run(async_driver(config, df_original.copy(), client))
    df_run = extract_data(df_run)
    df_run = df_run[df_run["label"] != "nan"].copy()

    df_run["score"] = pd.to_numeric(df_run["score"], errors="coerce")
    df_run["label"] = pd.to_numeric(df_run["label"], errors="coerce")

    df_run = df_run.dropna(subset=["score", "label"])

    df_run["correct"] = ((df_run["score"] - df_run["label"]).abs() <= 1).astype(int)

    overall_accuracy = df_run["correct"].mean()
    print(f"Overall accuracy: {overall_accuracy:.3f}")

    eval_df = (
        df_run.groupby("label")
        .agg(
            label_count=("label", "count"),
            correct_count=("correct", "sum")
        )
        .reset_index()
    )

    eval_df["accuracy"] = eval_df["correct_count"] / eval_df["label_count"]

    return df_run, eval_df

In [0]:
# echo but can add modifications to this function if desired
def enrich_text(df):
    return df

In [0]:
# load the data 
def load_data(path: str, config: dict = None) -> pd.DataFrame:
    df = pd.read_csv(path)
    df['Text'] = df['Text'].str.replace('/n', '', regex=False)

    if config['eval']:
        df['label'] = df['Consensus_Label']

    # First apply moving enrichment
    df = enrich_text(df)

    return df.astype(str)

In [0]:
config = {
        "response_format": get_schema(),
        "system_prompt": get_system_prompt(),
        "reasoning_effort": "low",
        "model_name": "gpt-5-mini",
        "semaphore_value": 10,
        "n_retries": 30,
        "eval": True,
    }

In [0]:
def LLM_driver(config = None):

    # Create a unique output directory based on date and time
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"./outputs/{timestamp}"
    os.makedirs(output_dir, exist_ok=True)

    df_original = load_data('./data/NPS_prepped.csv', config)

    if config['eval']:
        df_run, eval_df = eval(df_original.copy(), config)
        
        print("First Run Outputs")
        display(df_run)
        df_run.to_csv(f"{output_dir}/df_run_0.csv", index=False)

        print("Individual Run Data")
        display(eval_df)
        eval_df.to_csv(f"{output_dir}/df_eval.csv", index=False)

        return df_run, eval_df

    else:
        df_run = asyncio.run(async_driver(config, df_original.copy(), client))
        df_run.to_csv(f"{output_dir}/df_run_0.csv", index=False)
        df = extract_data(df_run)
        #display(df)

        # Return empty DataFrame for eval output
        empty_df = pd.DataFrame()
        return df, empty_df

In [0]:
df_run, df_eval = LLM_driver(config)
df_run.to_csv('./outputs/temp_run.csv', index=False)
df_eval.to_csv('./outputs/temp_eval.csv', index=False)
len(df_run)

100%|██████████| 107/107 [00:50<00:00,  2.12it/s]


Overall accuracy: 0.850
First Run Outputs


AgentRecordingSessionID Consensus_Label Certainty Text label predictions debug_info class score rationale citations debug_dict prompt_tokens cached_tokens completion_tokens total_tokens cache_rate correct 016EC1887C79794695583D66F37151EC 7 0.786 

1 Agent: Hello, my name is Jessette with Spectrum. I'm a specialist here in Texas. How May I Assist you today?

2 Caller: Umm are y'all having problems with your automated to where I can pay a Bill?

3 Agent: I have no idea.

4 Agent: If we are as of right now.

5 Agent: Right.

6 Caller: OK well I've tried 3 Times. They want a pin number that I don't know what my pin number is. In order for me to pay my Bill I gave him my account number.

7 Agent: Oh, OK.

8 Caller: My card number, my zip code, and it still tells me they want that pin number and I don't know what the heck it is. Almost said backward. I'm sorry that's your fault.

9 Agent: No, it's OK. So yeah, the automated system.

10 Caller: And they don't put it on the Bill anymore.

11 Agent: Right, Yeah, 'cause automated system will ask for the Security code and yes, we don't put it on the Bill no more. You can only find it on the app or on the website now.

12 Caller: Well when I went to the website it said my information is wrong.

13 Agent: OK.

14 Caller: And I don't have a car to drive down the spectrum to pay my Bill. I've always paid it on the telephone.

15 Agent: OK.

16 Caller: I'm a senior citizen.

17 Caller: And I'm just. I just can't.

18 Agent: Right.

19 Caller: Then I pay my Bill every month.

20 Caller: This is the first time it has asked me for my pin number which I don't know.

21 Agent: OK, now I understand. So let's take a look at the account. Verify your first and Last name and the address for me please.

22 Caller: Miracle Mystic.

23 Caller: 184.

24 Caller: Greenville Avenue Apartment B, Wilmington, North Carolina 28403.

25 Agent: OK. Thank you.

26 Agent: OK, and I mean we can try to update the Security code too as well. I could send you a temporary Security token to your phone number as a text if you like so we can upgrade your update your Security code.

27 Caller: Ok.

28 Caller: Yeah, well, I don't know if I changed my phone number or not.

29 Agent: OK 'cause.

30 Caller: I don't know if it's xxxx.

31 Caller: Or if it's 910 which is current.

32 Caller: xxxx.

33 Agent: 'Cause the only number we have listed is ending in 3251.

34 Agent: OK.

35 Caller: OK, well I changed my phone number. I went from Verizon to consumer cellular because I get Social Security and it's cheaper.

36 Agent: Right, OK.

37 Caller: Can you update that?

38 Agent: We can't update it right now 'cause I gotta get into the account. Do you know the Security question? It says what was the name of your first pet?

39 Caller: Pancho.

40 Agent: Pancho, OK.

41 Agent: Let's try.

42 Caller: It'll punch you a little bit.

43 Agent: Or a little bit, you said.

44 Caller: Yeah, or a little bit.

45 Agent: Is is that together or separate?

46 Caller: Together.

47 Agent: OK.

48 Agent: None of those worked.

49 Caller: I have not answered the questions in Forever.

50 Agent: All right, yeah, I won't be able to update the Security code, but I can at least take the payment.

51 Caller: For my cable, we shut off tomorrow, but I don't pay my Bill today.

52 Agent: It'll be shut off by the 6 if it's not paid by the 5th.

53 Agent: That's Fine. I mean I.

54 Caller: Yeah, but I have no transportation to go pay it and this is what I'm gonna have to go through.

55 Agent: Yeah. I mean, it's just I can take the payment for now and then unless you can.

56 Agent: With with your online account you can try to get into it, you just most likely have to.

57 Caller: It does it, doesn't it? It's wrong.

58 Agent: Well then that's why you click on forgot username or password and then you can retrieve your username and then reset the password.

59 Caller: Just went there.

60 Caller: And Fred because I had it to where it would remember it and then 

Individual Run Data


label,label_count,correct_count,accuracy
1,2,2,1.0
2,2,2,1.0
3,6,5,0.8333333333333334
4,2,1,0.5
5,8,5,0.625
6,17,8,0.47058823529411764
7,61,59,0.9672131147540983
8,8,8,1.0
9,1,1,1.0


107

In [0]:
print("Label set:", set(df_run['label'].unique()))
print("Class set:", set(df_run['class'].unique()))
print(set(df_run['class'].unique()).issubset(set(df_run['label'].unique())))

# Numerical table of class distribution
class_counts = df_run['label'].value_counts().reset_index()
class_counts.columns = ['label', 'count']
display(class_counts)

Label set: {np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)}
Class set: {'Detractor', 'Promoter', 'Passive'}
False


label,count
7,61
6,17
5,8
8,8
3,6
1,2
2,2
4,2
9,1
